# Generate All Derived Climate Metrics

This notebook generates ALL derived metrics (quantiles and day-count thresholds) from daily CHESS-SCAPE data.

**This must be run BEFORE `build_db.ipynb`** to ensure all required NetCDF files exist.

**Generated files:**
- Quantiles (99th percentile tasmax, 1st percentile tasmin) - annual, winter, summer
- Day-count metrics (tropical_nights, dry_days, hot_heat_days, heavy_rain_days, windy_days) - annual, winter, summer
- Both RCPs (60, 85)
- Both bias-corrected and non-bias-corrected
- Only decades 1980 and 2070

**Warning**: This downloads data from CEDA and can take several hours to complete.

In [1]:
import os
from pathlib import Path

import numpy as np
import xarray as xr
import yaml

import sys
sys.path.append(str(Path().resolve().parent / 'src'))  # Add data/src to sys.path

from process_daily_data import ClimateDataProcessor

In [2]:
# Run from repository root: research.LCAT.public
config_filepath = Path().resolve().parent / 'config.yml'
with open(config_filepath) as f:
    config = yaml.safe_load(f)

processor = ClimateDataProcessor(config, ensemble_member=1)
print('Data root:', config['chess_scape_netcdf_location'])

Data root: /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape


## Generate ALL derived metrics

This will generate:
- **Quantiles**: tasmax 99th, tasmin 1st for annual, winter, summer
- **Day-counts**: All 5 metrics (tropical_nights, dry_days, hot_heat_days, heavy_rain_days, windy_days)
- **For**: RCP 60 & 85, bias-corrected & non-bias-corrected, all seasons

**This will take several hours.** Progress will be shown below.

In [3]:
# Check what files already exist
def check_existing_files(config_dict):
    base = Path(config_dict['chess_scape_netcdf_location'])
    
    results = {
        'quantiles': {'missing': [], 'existing': []},
        'day_counts': {'missing': [], 'existing': []}
    }
    
    for rcp in [60, 85]:
        for bias in [True, False]:
            bias_suffix = '_bias-corrected' if bias else ''
            bias_label = 'bias' if bias else 'non-bias'
            
            for season in ['annual', 'winter', 'summer']:
                season_folder = 'annual' if season == 'annual' else 'seasonal'
                
                # Check quantiles
                for var, q in [('tasmax', 99), ('tasmin', 1)]:
                    filename = f'chess-scape_rcp{rcp}{bias_suffix}_01_{var}_{q}_percentile_uk_1km_{season}_19801201-20801130.nc'
                    path = base / f'data/rcp{rcp}{bias_suffix}/01/{season_folder}' / filename
                    label = f'RCP{rcp} {bias_label} {season} {var}_{q}th'
                    
                    if path.exists():
                        results['quantiles']['existing'].append(label)
                    else:
                        results['quantiles']['missing'].append((label, rcp, bias, season, var))
                
                # Check day-count metrics
                for metric in ['tropical_nights', 'dry_days', 'hot_heat_days', 'heavy_rain_days', 'windy_days']:
                    filename = f'chess-scape_rcp{rcp}{bias_suffix}_01_{metric}_uk_1km_{season}_19801201-20801130.nc'
                    path = base / f'data/rcp{rcp}{bias_suffix}/01/{season_folder}' / filename
                    label = f'RCP{rcp} {bias_label} {season} {metric}'
                    
                    if path.exists():
                        results['day_counts']['existing'].append(label)
                    else:
                        results['day_counts']['missing'].append((label, rcp, bias, season, metric))
    
    return results

results = check_existing_files(config)

print('=== QUANTILES ===')
print(f'Existing: {len(results["quantiles"]["existing"])}/36')
print(f'Missing: {len(results["quantiles"]["missing"])}/36')
if results['quantiles']['missing']:
    print('\nMissing quantile files:')
    for label, *_ in results['quantiles']['missing'][:10]:  # Show first 10
        print(f'  - {label}')
    if len(results['quantiles']['missing']) > 10:
        print(f'  ... and {len(results["quantiles"]["missing"]) - 10} more')

print('\n=== DAY-COUNT METRICS ===')
print(f'Existing: {len(results["day_counts"]["existing"])}/180')
print(f'Missing: {len(results["day_counts"]["missing"])}/180')
if results['day_counts']['missing']:
    print('\nMissing day-count files:')
    for label, *_ in results['day_counts']['missing'][:10]:  # Show first 10
        print(f'  - {label}')
    if len(results['day_counts']['missing']) > 10:
        print(f'  ... and {len(results["day_counts"]["missing"]) - 10} more')

# Determine what to generate
missing_rcps = set()
missing_bias = set()
missing_seasons = set()

for _, rcp, bias, season, _ in results['quantiles']['missing'] + results['day_counts']['missing']:
    missing_rcps.add(rcp)
    missing_bias.add(bias)
    missing_seasons.add(season)

print('\n=== WHAT TO GENERATE ===')
print(f'RCPs: {sorted(missing_rcps) if missing_rcps else "None - all complete!"}')
print(f'Bias options: {sorted(missing_bias) if missing_bias else "None - all complete!"}')
print(f'Seasons: {sorted(missing_seasons) if missing_seasons else "None - all complete!"}')

=== QUANTILES ===
Existing: 0/36
Missing: 24/36

Missing quantile files:
  - RCP60 bias annual tasmax_99th
  - RCP60 bias annual tasmin_1th
  - RCP60 bias winter tasmax_99th
  - RCP60 bias winter tasmin_1th
  - RCP60 bias summer tasmax_99th
  - RCP60 bias summer tasmin_1th
  - RCP60 non-bias annual tasmax_99th
  - RCP60 non-bias annual tasmin_1th
  - RCP60 non-bias winter tasmax_99th
  - RCP60 non-bias winter tasmin_1th
  ... and 14 more

=== DAY-COUNT METRICS ===
Existing: 4/180
Missing: 56/180

Missing day-count files:
  - RCP60 bias annual tropical_nights
  - RCP60 bias annual dry_days
  - RCP60 bias annual hot_heat_days
  - RCP60 bias annual heavy_rain_days
  - RCP60 bias annual windy_days
  - RCP60 bias winter hot_heat_days
  - RCP60 bias winter heavy_rain_days
  - RCP60 bias winter windy_days
  - RCP60 bias summer hot_heat_days
  - RCP60 bias summer heavy_rain_days
  ... and 46 more

=== WHAT TO GENERATE ===
RCPs: [60, 85]
Bias options: [False, True]
Seasons: ['annual', 'summer',

## Generate only missing files

Based on the check above, this will generate only what's missing. Adjust the parameters below if needed.

In [ ]:
# Generate only missing files - all metrics enabled
# This will process efficiently by calculating ALL metrics for each variable's data

if missing_rcps or missing_bias or missing_seasons:
    print(f'Generating for: RCPs={sorted(missing_rcps)}, bias={sorted(missing_bias)}, seasons={sorted(missing_seasons)}')
    
    processor.generate_data(
        quantiles_config={},
        tropical_nights_enabled=True,
        hot_days_enabled=True,
        heavy_rain_enabled=True,
        dry_days_enabled=True,
        windy_days_enabled=True,
        rcps=sorted(missing_rcps) if missing_rcps else [60, 85],
        bias_options=sorted(missing_bias) if missing_bias else [True, False],
        seasons=sorted(missing_seasons) if missing_seasons else ['annual', 'winter', 'summer'],
        variables=['pr', 'tasmin', 'tasmax', 'sfcWind'],
    )
    
    print('\n✓ All derived metrics complete!')
else:
    print('All files already exist! Nothing to generate.')

Generating for: RCPs=[60, 85], bias=[False, True], seasons=['annual', 'summer', 'winter']
Processing RCP 60, Bias Corrected: False, Variable: tasmin
Getting grid dimensions...
Getting grid dimensions...
Grid size: 1057 x 656
Filtering files for season: annual
Files after season filtering: 1200 out of 1200
Files per decade: [(0, 120), (9, 120)]

Processing step-decade 0 (120 files)...
Grid size: 1057 x 656
Filtering files for season: annual
Files after season filtering: 1200 out of 1200
Files per decade: [(0, 120), (9, 120)]

Processing step-decade 0 (120 files)...


Loading files:  27%|██▋       | 32/120 [04:24<12:00,  8.19s/it] 

Retry 1/4 for chess-scape_rcp60_01_tasmin_uk_1km_daily_19830201-19830230.nc after 2.0s delay


Loading files:  42%|████▎     | 51/120 [06:34<06:39,  5.80s/it]

Retry 1/4 for chess-scape_rcp60_01_tasmin_uk_1km_daily_19840501-19840530.nc after 1.4s delay


Loading files:  54%|█████▍    | 65/120 [07:13<06:06,  6.66s/it]



In [ ]:
processor.generate_data(
    quantiles_config={'tasmax': [99], 'tasmin': [1]},
    tropical_nights_enabled=True,
    hot_days_enabled=True,
    heavy_rain_enabled=True,
    dry_days_enabled=True,
    windy_days_enabled=True,
    rcps=[60, 85],
    bias_options=[True, False],
    seasons=['annual', 'winter', 'summer'],
    variables=['pr', 'tasmin', 'tasmax', 'sfcWind'],
    tropical_threshold=20.0,
    heat_threshold=30.0,
    rain_threshold=50.0,
    dry_threshold=1.0,
    wind_threshold=8.0,
)

Processing RCP 60, Bias Corrected: True, Variable: pr
Getting grid dimensions...
Getting grid dimensions...
Grid size: 1057 x 656
Filtering files for season: annual
Files after season filtering: 1200 out of 1200
Files per decade: [(0, 120), (9, 120)]

Processing step-decade 0 (120 files)...
Grid size: 1057 x 656
Filtering files for season: annual
Files after season filtering: 1200 out of 1200
Files per decade: [(0, 120), (9, 120)]

Processing step-decade 0 (120 files)...


Loading files:  38%|███▊      | 46/120 [08:20<19:33, 15.86s/it]

Retry 1/4 for chess-scape_rcp60_bias-corrected_01_pr_uk_1km_daily_19840201-19840230.nc after 1.5s delay


Loading files: 100%|██████████| 120/120 [19:22<00:00,  9.69s/it]



Successfully processed 120/120 files
Step-decade 0 data shape: (3600, 1057, 656)
Applying calculation for step-decade 0...
Step-decade 0 data shape: (3600, 1057, 656)
Applying calculation for step-decade 0...


: 

In [ ]:
def expected_output_path(config_dict, rcp, bias_corrected, ensemble_member, metric, season):
    base = Path(config_dict['chess_scape_netcdf_location'])
    bias_suffix = '_bias-corrected' if bias_corrected else ''
    season_folder = 'annual' if season == 'annual' else 'seasonal'
    ensemble_str = f'{ensemble_member:02d}'
    filename = (
        f'chess-scape_rcp{rcp}{bias_suffix}_{ensemble_str}_{metric}_'
        f'uk_1km_{season}_19801201-20801130.nc'
    )
    return base / f'data/rcp{rcp}{bias_suffix}/{ensemble_str}/{season_folder}' / filename

# Check a sample of generated files
test_files = [
    ('RCP60 bias summer dry_days', 60, True, 'dry_days', 'summer'),
    ('RCP60 non-bias winter tropical', 60, False, 'tropical_nights', 'winter'),
    ('RCP85 bias annual hot_heat_days', 85, True, 'hot_heat_days', 'annual'),
    ('RCP85 non-bias summer heavy_rain', 85, False, 'heavy_rain_days', 'summer'),
    ('RCP60 bias winter quantile', 60, True, 'tasmin_1_percentile', 'winter'),
]

print('Checking sample files:')
for label, rcp, bias, metric, season in test_files:
    path = expected_output_path(config, rcp, bias, 1, metric, season)
    exists = '✓' if path.exists() else '✗'
    print(f'  {exists} {label}')

summer_dry /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_summer_19801201-20801130.nc
  exists: True
winter_dry /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_winter_19801201-20801130.nc
  exists: True
summer_tropical /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
  exists: True
winter_tropical /home/cat/Desktop/projects/lcat/research.LCAT.public/data/store/chess-scape/data/rcp60_bias-corrected/01/seasonal/chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_winter_19801201-20801130.nc
  exists: True


In [ ]:
# Inspect a few files to verify correctness
def inspect_day_count_file(path, season, metric_name):
    if not path.exists():
        print(f'--- {metric_name} ---')
        print(f'File does not exist: {path.name}')
        return
    
    ds = xr.open_dataset(path, engine='netcdf4')
    try:
        data = ds['variable']
        arr = data.values

        print('---', metric_name, '---')
        print('file:', path.name)
        print('dims:', data.dims)
        print('shape:', arr.shape)
        print('decades:', ds.decade.values)
        print('min:', float(np.nanmin(arr)))
        print('mean:', float(np.nanmean(arr)))
        print('max:', float(np.nanmax(arr)))

        # Validate bounds
        upper_bound = 360 if season == 'annual' else 90
        max_val = float(np.nanmax(arr))
        if max_val <= upper_bound + 1e-6:
            print(f'✓ Max value {max_val:.1f} is within expected bound of {upper_bound}')
        else:
            print(f'✗ WARNING: Max value {max_val:.1f} exceeds bound of {upper_bound}')
        print()
    finally:
        ds.close()

# Test a few different files
summer_dry = expected_output_path(config, 60, True, 1, 'dry_days', 'summer')
annual_tropical = expected_output_path(config, 85, False, 1, 'tropical_nights', 'annual')
winter_hot = expected_output_path(config, 60, True, 1, 'hot_heat_days', 'winter')

inspect_day_count_file(summer_dry, 'summer', 'RCP60 bias dry_days summer')
inspect_day_count_file(annual_tropical, 'annual', 'RCP85 non-bias tropical_nights annual')
inspect_day_count_file(winter_hot, 'winter', 'RCP60 bias hot_heat_days winter')

--- dry_days_summer ---
file: chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_summer_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 18.732587915637897
max: 72.9
--- dry_days_winter ---
file: chess-scape_rcp60_bias-corrected_01_dry_days_uk_1km_winter_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 14.334348247455985
max: 60.3
--- tropical_nights_summer ---
file: chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_summer_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 0.16198542815607908
max: 28.9
--- tropical_nights_winter ---
file: chess-scape_rcp60_bias-corrected_01_tropical_nights_uk_1km_winter_19801201-20801130.nc
dims: ('decade', 'y', 'x')
shape: (2, 1057, 656)
decades: [0 9]
min: 0.0
mean: 0.0
max: 0.0


## Summary

If the checks above pass, all derived metrics have been successfully generated and are ready for database loading.

You can now run `build_db.ipynb` to load all data (base variables + derived metrics) into PostgreSQL.